In [ ]:
import json
import os
import numpy as np
import pandas as pd

# --- HELPER FUNCTION TO CLEAN NUMPY TYPES FOR JSON ---
def convert_to_serializable(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient="records")
    return obj

print("Compiling dashboard metrics from memory...")

# =====================================================================
# 1. KPI DATA DECLARATIONS
# (Replace placeholders with your actual notebook variable names)
# =====================================================================
kpi_payload = {
    "childrenTested": convert_to_serializable(df_ml.shape[0]), 
    "malariaPrevalence": float(round((df_ml['hml32_pos'].mean() * 100), 1)), # Assuming 'hml32_pos' is binary target
    "modelAccuracy": float(round(xgb_accuracy * 100, 1)),                  # Your XGBoost accuracy variable
    "aucRoc": float(round(xgb_auc_score, 2)),                               # Your XGBoost AUC variable
    "r0": float(round(calculated_r0, 2)),                                   # Your SEIR baseline R0
    "regionsAnalyzed": int(df_ml['region'].nunique())                      # Unique regional scope
}

# =====================================================================
# 2. REGIONAL RISK DATA EXTRACTION
# Assumes you have a regional summary DataFrame. If your columns have 
# different names, map them inside the lambda or rename them here.
# =====================================================================
# Example source columns: 'region', 'risk_score', 'prevalence', 'latitude', 'longitude'
regional_list = []
for _, row in df_regional_summary.iterrows():
    # Dynamic classification logic matching your tier rules
    prevalence_val = row['prevalence']
    if prevalence_val >= 30.0:
        tier_str = "Critical"
    elif prevalence_val >= 20.0:
        tier_str = "High"
    elif prevalence_val >= 10.0:
        tier_str = "Moderate"
    else:
        tier_str = "Low"

    regional_list.append({
        "region": str(row['region']),
        "riskScore": float(round(row['risk_score'], 2)),
        "prevalence": float(round(prevalence_val, 1)),
        "tier": tier_str,
        "lat": float(row['latitude']),
        "lng": float(row['longitude'])
    })

# =====================================================================
# 3. FEATURE IMPORTANCE EXTRACTION
# Assumes a DataFrame with feature names and importance weights
# =====================================================================
# Mapping clean UI presentation text over short dataset labels if needed
feature_mapping = {
    "hml32_thatch": "Thatch/Straw Roof",
    "hv270_wealth": "Wealth Index",
    "hml32_iron": "Iron Sheet Roof"
}

feature_list = []
for _, row in df_feature_importance.iterrows():
    raw_name = row['feature']
    feature_list.append({
        "feature": feature_mapping.get(raw_name, raw_name),
        "importance": float(round(row['importance'], 2)),
        "category": str(row['category']) # e.g., 'Housing', 'Socioeconomic'
    })

# =====================================================================
# 4. EVALUATION MATRICES (Confusion Matrix, Curves & Comparisons)
# =====================================================================
# Confusion Matrix breakdown
confusion_payload = {
    "truePositive": int(tn_fp_fn_tp_array[1][1]), # adjust index array based on your confusion matrix output
    "falsePositive": int(tn_fp_fn_tp_array[0][1]),
    "falseNegative": int(tn_fp_fn_tp_array[1][0]),
    "trueNegative": int(tn_fp_fn_tp_array[0][0])
}

# Model Comparison Array
model_comparison_list = [
    {"model": "XGBoost", "accuracy": float(xgb_acc), "auc": float(xgb_auc), "precision": float(xgb_prec), "recall": float(xgb_rec), "f1": float(xgb_f1)},
    {"model": "Random Forest", "accuracy": float(rf_acc), "auc": float(rf_auc), "precision": float(rf_prec), "recall": float(rf_rec), "f1": float(rf_f1)},
    {"model": "Logistic Reg.", "accuracy": float(lr_acc), "auc": float(lr_auc), "precision": float(lr_prec), "recall": float(lr_rec), "f1": float(lr_f1)}
]

# ROC Curve Threshold Coordinates (Assuming data generated by sklearn roc_curve)
# fpr_array, tpr_array = roc_curve(y_test, y_probs)
roc_curve_list = [{"fpr": float(round(f, 2)), "tpr": float(round(t, 2))} for f, t in zip(fpr_array, tpr_array)]

# Precision-Recall Tradeoff Metrics
# precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs)
pr_tradeoff_list = [{"threshold": float(round(thresh, 1)), "precision": float(round(p, 2)), "recall": float(round(r, 2))} 
                    for p, r, thresh in zip(precisions, recalls, thresholds)]


# =====================================================================
# 5. COMPILE AND WRITE THE TARGET ARTIFACT
# =====================================================================
master_payload = {
    "kpiData": kpi_payload,
    "regionalData": regional_list,
    "featureImportanceData": feature_list,
    "confusionMatrix": confusion_payload,
    "modelComparison": model_comparison_list,
    "rocCurveData": roc_curve_list,
    "precisionRecallData": pr_tradeoff_list
}

# Define your relative path targeting your React Vite directory structure
output_directory = "../frontend-project/src/data"
output_file_path = os.path.join(output_directory, "dashboardData.json")

os.makedirs(output_directory, exist_ok=True)
with open(output_file_path, "w", encoding="utf-8") as json_file:
    json.dump(master_payload, json_file, indent=2, default=convert_to_serializable)

print(f"Data engine output synced successfully to: {output_file_path}")